<a href="https://colab.research.google.com/github/anuradha-gh/AI-Powered-Granular-Access-Control-for-SaaS-Applications-/blob/Explainability-Engine/Explainability_Engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:


# @title 1. Setup and Installations
!pip install transformers[torch] datasets scikit-learn pandas faiss-cpu -q

import pandas as pd
import re
import os
from google.colab import drive
from transformers import AutoTokenizer, AutoModel
import torch
# import faiss but will not use it yet
import faiss
import numpy as np

print("Libraries installed successfully.")
print("NOTE: This notebook conceptually requires a model from Component 2.")
print("For this demo, we will use a generic pre-trained model as a placeholder.")

# @title 2. Load Data and Embedding Model
# --- Configuration ---
# For this initial demo, we don't need the *fine-tuned* C2 model yet.
# We can use a generic, pre-trained "Sentence-BERT" model which is
# excellent for creating embeddings.
EMBEDDING_MODEL_CHECKPOINT = 'sentence-transformers/all-MiniLM-L6-v2'

# We will use the real, large dataset for our RAG knowledge base
DATA_PATH = '/content/drive/MyDrive/cybersecurity_threat_detection_logs.csv'
# ---------------------

try:
    drive.mount('/content/drive')

    # Load a generic Sentence-BERT model
    print(f"Loading tokenizer from {EMBEDDING_MODEL_CHECKPOINT}...")
    tokenizer = AutoTokenizer.from_pretrained(EMBEDDING_MODEL_CHECKPOINT)

    print(f"Loading model from {EMBEDDING_MODEL_CHECKPOINT}...")
    model = AutoModel.from_pretrained(EMBEDDING_MODEL_CHECKPOINT)

    # Load the dataset
    print(f"Loading data from {DATA_PATH}...")
    df = pd.read_csv(DATA_PATH)

    # Get one sample row for the demo
    demo_row = df.sample(n=1).iloc[0]

    print(f"Loaded {len(df)} records. Will use one sample for this demo.")

except Exception as e:
    print(f"--- ERROR ---")
    print(f"Could not load data or model. Please check your paths.")
    print(f"Error details: {e}")
    model = None


# @title 3. Log-to-Sentence Transformation
# We use a similar function to C2/C3 to make the logs human-readable
# and embeddable for the RAG system.
def log_to_rag_sentence(row):
    def sanitize(s):
        return re.sub(r'[^a-zA-Z0-9_./-]', '', str(s))

    return (
        f"At {row.get('timestamp', 'unknown time')}, "
        f"source IP {sanitize(row.get('source_ip', '?'))} "
        f"used {sanitize(row.get('protocol', '?'))} "
        f"to access path '{sanitize(row.get('request_path', '?'))}' "
        f"with user agent '{sanitize(row.get('user_agent', '?'))}'. "
        f"The action was '{sanitize(row.get('action', '?'))}' "
        f"and flagged as '{sanitize(row.get('threat_label', '?'))}'."
    )

if model:
    print("Converting one log row to a sentence...")
    sentence = log_to_rag_sentence(demo_row)
    print("\n--- Example RAG Sentence ---")
    print(sentence)


# @title 4. Demonstrate Single Embedding Generation
# This is the core of the "Indexing" process
def get_embedding(sentence):
    """
    Uses the loaded model to generate a single embedding.
    We take the mean pooling of the last hidden state.
    """
    # Tokenize the sentence
    inputs = tokenizer(
        sentence,
        padding=True,
        truncation=True,
        return_tensors="pt",
        max_length=128
    )

    # Get model output
    with torch.no_grad():
        outputs = model(**inputs)

    # Perform mean pooling
    # Take the average of all token embeddings in the last hidden state
    embedding = outputs.last_hidden_state.mean(dim=1)
    return embedding

if 'sentence' in locals():
    print("Generating a semantic embedding for the example sentence...")

    embedding_vector = get_embedding(sentence)

    print("Embedding generation complete.")
    print(f"  Vector Shape: {embedding_vector.shape}")
    print(f"  Example Vector (first 5 dimensions): {embedding_vector[0, :5]}")


# @title 5. Show Results and Next Steps
if 'embedding_vector' in locals():




SyntaxError: incomplete input (ipython-input-3446027954.py, line 117)